In [ ]:
# import libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.colors import TABLEAU_COLORS

from cycler import cycler
from scipy.integrate import solve_ivp
from scipy.special import legendre, legendre_p
import pysindy as ps

# import modules from parent dir
# from <https://stackoverflow.com/questions/714063/importing-modules-from-parent-folder>
import os
# below no worki, maybe because of ipynb?
# import sys
# import inspect
# currentdir = os.path.dirname(os.path.abspath(inspect.getfile(inspect.currentframe())))
# print(currentdir)
# parentdir = os.path.dirname(currentdir)
# print(parentdir)
# sys.path.insert(0, parentdir)

currentdir = os.getcwd()
if currentdir.split("/")[-1] != 'yukawa-sindy':
    parentdir = os.path.dirname(currentdir)
    os.chdir(parentdir)

import Yukawa_SINDy as ys
import cross_validation as cv
import pickle as pkl
import anisotropic_potential as ap
from importlib import reload

# import Yukawa scaling constant
with open('scaling_const.float', 'rb') as f:
    SCALING_CONST = pkl.load(f)

# import Mach number estimateion
with open('mach.float', 'rb') as f:
    MACH_NUM = pkl.load(f)

plt.rcParams.update({'font.size': 12})
plt.rcParams.update({'figure.figsize': (8,6)})

In [ ]:
reload(ap)

## Plot of Legendre Polynomials of cosine argument and derivatives

In [ ]:
order = 2
theta = np.r_[0:np.pi:100j]
x = np.cos(theta)
y = legendre_p(order, x, diff_n = 2).T

fig, axd = plt.subplot_mosaic("AB", per_subplot_kw={"A": {'projection': 'polar'}},figsize=(10,5))
for key in axd.keys():
    axd[key].plot(theta, y, label = [f"$P_{order}(\\cos\\theta)$", f"$P_{order}'(\\cos\\theta))$", f"$P_{order}''(\\cos\\theta)$"])
    axd[key].set_xticks([0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi])
    axd[key].set_xticklabels(['0', '$\\frac{\\pi}{4}$', '$\\frac{\\pi}{2}$', '$\\frac{3\\pi}{4}$', '$\\pi$'])
    axd[key].legend()

# individual plot formatting
axd['A'].set_thetamax(180)
axd['A'].axhline(c='black', lw=1, ls='--')

axd['B'].set_xlabel("$\\theta$")

fig.suptitle(f"Legendre polynomial of order {order} and derivatives")
fig. tight_layout()

The Kompaneets anisotropic potential I am using here is the same one from Ivlev, *et al.* (2010) "Electrorheological Complex Plasmas." It comes from Kompaneets' dissertation, and can be expressed as

\begin{equation}
    \frac{V}{V_0} = 
    \frac{e^{-r}}{r} - (4-\pi) \frac{M^2}{r^3} P_2(\cos\theta),
\end{equation}

where $V_0 = q_d^2/(4\pi\varepsilon_0\lambda_{Di})$, $\lambda_{Di}$ is the ion Debye length, $P_2$ is the second order Legendre polynomial, $M$ is the ion drift Mach number, $r$ is the radius normalized to the ion Debye length, and $\theta$ is the angle with respect to the external electric field (horizontal).

## Differences betwixt scipy `legendre_p` and explicitly writing out legendre algebraically

In [ ]:
# Make data for comparing scipy to explicit legendre polynomials
test_data = np.linspace(-1,1,1000, endpoint=False)

# generate test data
P_2_scipy = lambda x: legendre_p(2,x)[0]
scipy_test = P_2_scipy(test_data)
P_2_explicit = lambda x: (1/2) * (3*x**2 - 1)
explicit_test = P_2_explicit(test_data)

In [ ]:
mask = scipy_test != explicit_test
# Check how many differences between scipy legendre and explicit legendre there are
np.count_nonzero(mask)

In [ ]:
# check size of differences
diffs = np.abs(scipy_test[mask]-explicit_test[mask])
diffs.max()

In [ ]:
custom_cycler = (cycler(color=TABLEAU_COLORS)[:-2] +
                  2*cycler(linestyle=['-', '--', ':', '-.']))
fig, axs = plt.subplots(1,2,figsize=(10,5))
all_test = np.vstack((scipy_test, explicit_test)).T
axs[0].set_prop_cycle(custom_cycler)
axs[0].plot(test_data, all_test, label = ["scipy", "explicit"], lw=3)
axs[0].set_xlabel("$x$")
axs[0].set_ylabel("$P_2(x)$")
axs[0].legend()

all_diffs = np.abs(scipy_test - explicit_test)
axs[1].hist(all_diffs)
axs[1].set_xlabel("absolute difference")
axs[1].set_ylabel("count")

## Kompaneets potential plot

In [ ]:
# Kompaneets potential
def anisotropic_potential(r, theta, M=MACH_NUM):
    P_2 = lambda x: legendre_p(2, x)[0]
    return np.exp(-r) / r - (4 - np.pi) * (M**2 / r**3)  * P_2(np.cos(theta))

In [ ]:
ptcl_radius = 0.0239 # in ion Debye lengths

In [ ]:
r = np.linspace(ptcl_radius,1,5000)
theta = np.linspace(0,2*np.pi,1440)
R, Theta = np.meshgrid(r, theta)
potential = anisotropic_potential(R, Theta)#, M=3e-1)
# x and y are bounds, so potential should be the value *inside* those bounds.
# Therefore, remove the last value from the potential array.
potential = potential[:-1, :-1]

# plot
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(15,10),subplot_kw={'projection': 'polar'})

# linear scale
ax1.set_title("Linear")
im = ax1.pcolormesh(Theta, R, potential, norm=colors.CenteredNorm(), cmap='coolwarm')
ax1.set_rmax(0.08)
# colorbar
colorbar_pad = 0.15
shrink = 0.5
fig.colorbar(im, ax=ax1, label='$V/V_0$', pad=colorbar_pad, shrink=shrink)

# log scale
ax2.set_title("Logarithmic")
Norm = colors.SymLogNorm(linthresh=1, linscale=1, vmin=-5e3, vmax=5e3)
im = ax2.pcolormesh(Theta, R, potential, norm=Norm, cmap='coolwarm')
# colorbar
fig.colorbar(im, ax=ax2, label='$V/V_0$', pad=colorbar_pad, shrink=shrink)

fig.suptitle("Normalized Kompaneets potential", y=0.85)
fig.tight_layout()

In [ ]:
10//4

In [ ]:
r_max = 10
r = np.linspace(ptcl_radius,r_max,1000)
theta = np.linspace(0,2*np.pi,1440)
R, Theta = np.meshgrid(r, theta)
potential = anisotropic_potential(R, Theta)#, M=0.3)
# x and y are bounds, so potential should be the value *inside* those bounds.
# Therefore, remove the last value from the potential array.
potential = potential[:-1, :-1]
fig = plt.figure()
ax = plt.subplot(projection='polar')
ax.set_title("Normalized Kompaneets potential")
Norm = colors.SymLogNorm(linthresh=1e-3, linscale=1)#, vmin=-5e3, vmax=5e3)
im = ax.pcolormesh(Theta, R, potential, norm=Norm, cmap='coolwarm')
step = r_max//4
start = step
stop = r_max+step
ax.set_rticks(np.arange(start, stop, step))
fig.colorbar(im, ax=ax, label='$\\tilde{V}$')
fig.tight_layout()
fig.set_dpi(300)

In [ ]:
MACH_NUM

In [ ]:
test = legendre_p(order, x, diff_n = 2)
test.shape

## Simulation testing

In [ ]:
reload(ap)
# rng=np.random.default_rng(seed=34123)
rng=None
sim = ap.Anisotropic_simulation(rng=rng)
duration = 3
# duration = 0.0913 is so far the longest duration that won't take forever 
# for initial conditions [0.5, 1, np.pi/8, -1]
dt = 1e-3
r0 = 10
p0 = -5
theta0 = -np.pi/4
l0 = -5
x0 = np.array([r0, p0, theta0, l0])
sim.simulate(duration, dt, x0=x0)
fig, axs = sim.plot()

In [ ]:
sliced = sim.r1[:30]
sliced.shape

In [ ]:
sim.r1[0]

In [ ]:
sim.r1.shape